In [4]:
import requests

print("Githubからのダウンロードを開始します")

url = "https://raw.githubusercontent.com/aiknowledgelearning-decuments/document01/main/section11/ank.db"
r = requests.get(url)

with open("ank.db", "wb") as f:
    f.write(r.content)

print("Githubからのダウンロードが完了しました")


Githubからのダウンロードを開始します
Githubからのダウンロードが完了しました


In [5]:
# Colabで config.py を作る（同じ作業ディレクトリに作成）
api_key = "XXXXXXXXXX"  # ←自分のキーに置き換え

with open("config.py", "w", encoding="utf-8") as f:
    f.write(f'OPENAI_API_KEY = "{api_key}"\n')

print("config.py を作成しました")

config.py を作成しました


In [13]:
import sqlite3
import json
import time
from openai import OpenAI
import config

DB_PATH = "ank.db"
MODEL_NAME = "gpt-4.1-mini"
CHUNK_SIZE = 500
client = OpenAI(api_key=config.OPENAI_API_KEY)

CATEGORY_LIST = ["policy", "definition", "fact", "context", "noise"]

SYSTEM_PROMPT = """
あなたはQAデータをナレッジ化するための分類担当です。
以下のQAを、必ず次の5カテゴリのいずれかに分類してください。
未分類、NULL、その他は禁止です。

カテゴリ定義:

noise:
挨拶、終了発言、中断、確認だけの発言、謝罪、感想、抽象的な意見など、
意味が薄くナレッジ化に向かないQA
context:
会議進行、発言者、出席者、手続き、質疑の順番、誰が何を確認したかなど、
その会議の文脈確認に関するQA
definition:
用語、仕組み、制度、法律、概念の説明に関するQA
fact:
具体的な事実、数値、時期、対象、場所、出来事、実施状況に関するQA
policy:
制度、政策、施策、方針、支援策、計画、政府の対応方針に関するQA

優先順位:
1. noise
2. context
3. definition
4. fact
5. policy

判定ルール:
- 挨拶、謝罪、確認だけ、意味が薄い発言は noise
- 会議の進行、発言者、出席者、順番、異議確認は context
- 用語や制度の意味を説明している場合は definition
- 数値、日時、場所、対象、出来事などの客観情報は fact
- 政策判断、制度変更、支援策、計画、政府方針は policy
- policy、definition、fact はナレッジ化対象
- context、noise はナレッジ化対象外
- 理由は短く、1文で書いてください
- 出力は必ずJSONのみ

出力形式:
{
  "category": "context",
  "is_knowledge_target": 0,
  "reason": "会議進行に関する内容のため"
}
"""

def classify_qa(question, answer):
    user_prompt = f"""
質問:
{question}

回答:
{answer}
"""

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0
    )

    content = response.choices[0].message.content.strip()
    return json.loads(content)

conn = sqlite3.connect(DB_PATH)
conn.row_factory = sqlite3.Row
cursor = conn.cursor()

cursor.execute("""
SELECT COUNT(*)
FROM qa
WHERE qa_category IS NULL
   OR qa_category = ''
""")
remain_count = cursor.fetchone()[0]

print(f"未分類件数: {remain_count}")

if remain_count == 0:
    print("すべて分類済みです。")
    conn.close()
    raise SystemExit()

cursor.execute("""
SELECT qa_id, question, answer
FROM qa
WHERE qa_category IS NULL
   OR qa_category = ''
ORDER BY qa_id
LIMIT ?
""", (CHUNK_SIZE,))

rows = cursor.fetchall()
print(f"今回分類件数: {len(rows)}")

success_count = 0
error_count = 0

for idx, row in enumerate(rows, start=1):
    qa_id = row["qa_id"]
    question = row["question"] or ""
    answer = row["answer"] or ""

    try:
        result = classify_qa(question, answer)

        category = result.get("category")
        reason = result.get("reason", "")
        is_knowledge_target = int(result.get("is_knowledge_target", 0))

        if category not in CATEGORY_LIST:
            raise ValueError(f"不正なカテゴリです: {category}")

        if category in ["policy", "definition", "fact"]:
            is_knowledge_target = 1
        else:
            is_knowledge_target = 0

        cursor.execute("""
        UPDATE qa
           SET qa_category = ?,
               qa_category_reason = ?,
               is_knowledge_target = ?
         WHERE qa_id = ?
        """, (
            category,
            reason,
            is_knowledge_target,
            qa_id
        ))

        success_count += 1

        if idx % 20 == 0:
            conn.commit()
            print(f"{idx}件完了")

    except Exception as e:
        error_count += 1
        print(f"ERROR qa_id={qa_id}: {e}")

    time.sleep(0.1)

conn.commit()

print("=================================")
print("カテゴリ分類処理完了")
print(f"成功件数: {success_count}")
print(f"失敗件数: {error_count}")

cursor.execute("""
SELECT qa_category, is_knowledge_target, COUNT(*)
FROM qa
GROUP BY qa_category, is_knowledge_target
ORDER BY qa_category
""")

for row in cursor.fetchall():
    print(f"{row[0]} / target={row[1]} : {row[2]}件")

cursor.execute("""
SELECT COUNT(*)
FROM qa
WHERE qa_category IS NULL
   OR qa_category = ''
""")
remain_after = cursor.fetchone()[0]

print(f"残件数: {remain_after}")

if remain_after > 0:
    print("CHUNK_SIZE到達のため途中終了しました。再実行してください。")
else:
    print("全件分類が完了しました。")

conn.close()

未分類件数: 0
すべて分類済みです。


SystemExit: 

/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
